In [ ]:
from pathlib import Path
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from PIL import Image
import ipywidgets as widgets
from IPython.display import display, clear_output

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

PROCESSED = ROOT / "data" / "processed"
ANNOTATIONS = ROOT / "data" / "annotations"
VIEWS = PROCESSED / "perspective_views"

ANNOTATIONS.mkdir(parents=True, exist_ok=True)

locations = gpd.read_file(
    PROCESSED / "panorama_locations.geojson"
)

metadata = pd.read_csv(
    PROCESSED / "perspective_views_metadata.csv"
)

print("Locations:", len(locations))
print("Images:", len(metadata))

In [ ]:
import numpy as np

gdf = locations.to_crs("EPSG:28992").copy()

# Reproducible random generator
rng = np.random.default_rng(42)

# Divide the study area into a 4 × 4 grid
gdf["grid_x"] = pd.cut(
    gdf.geometry.x,
    bins=4,
    labels=False,
    include_lowest=True
)

gdf["grid_y"] = pd.cut(
    gdf.geometry.y,
    bins=4,
    labels=False,
    include_lowest=True
)

gdf["stratum"] = (
    gdf["grid_x"].astype(str)
    + "_"
    + gdf["grid_y"].astype(str)
)

# Randomly select one location per occupied cell
selected_ids = []

for _, group in gdf.groupby("stratum"):
    selected_ids.append(
        rng.choice(group["location_id"].to_numpy())
    )

# Fill remaining slots randomly
remaining_ids = gdf.loc[
    ~gdf["location_id"].isin(selected_ids),
    "location_id"
].to_numpy()

additional = rng.choice(
    remaining_ids,
    size=20 - len(selected_ids),
    replace=False
)

selected_ids = sorted(
    selected_ids + additional.tolist()
)

annotation_images = (
    metadata[
        metadata["location_id"].isin(selected_ids)
    ]
    .sort_values(["location_id", "image_id"])
    .reset_index(drop=True)
)

assert len(annotation_images) == 40
assert annotation_images["location_id"].nunique() == 20

print("Selected locations:", selected_ids)
print("Images for annotation:", len(annotation_images))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))

gdf.plot(
    ax=ax,
    color="lightgray",
    markersize=25,
    label="All 50 locations"
)

gdf[gdf["location_id"].isin(selected_ids)].plot(
    ax=ax,
    color="royalblue",
    markersize=55,
    label="Selected 20 locations"
)

ax.legend()
ax.set_title("Human annotation sample")
plt.show()

In [ ]:
annotation_images.to_csv(
    ANNOTATIONS / "annotation_sample.csv",
    index=False
)

In [ ]:
ANNOTATION_FILE = ANNOTATIONS / "human_annotations.csv"

FIELDS = {
    "greenery": ["low", "medium", "high", "unclear"],
    "pedestrian": ["weak", "moderate", "strong", "unclear"],
    "cycling": ["absent", "present", "unclear"],
    "motor_vehicle": ["low", "medium", "high", "unclear"]
}

# Load previously saved annotations, if any
if ANNOTATION_FILE.exists():
    saved = pd.read_csv(
        ANNOTATION_FILE,
        keep_default_na=False
    )
else:
    saved = pd.DataFrame(
        columns=["image_id", "location_id", *FIELDS, "notes"]
    )

current_index = 0

image_output = widgets.Output()
status = widgets.HTML()

controls = {
    name: widgets.Dropdown(
        options=["-- select --"] + choices,
        description=name.replace("_", " ").title() + ":",
        style={"description_width": "150px"},
        layout=widgets.Layout(width="370px")
    )
    for name, choices in FIELDS.items()
}

notes = widgets.Textarea(
    description="Notes:",
    placeholder="Optional: ambiguity, obstruction, unusual features...",
    style={"description_width": "150px"},
    layout=widgets.Layout(width="500px", height="75px")
)

save_button = widgets.Button(
    description="Save & Next",
    button_style="success"
)

def show_image():
    row = annotation_images.iloc[current_index]

    with image_output:
        clear_output(wait=True)
        display(
            Image.open(VIEWS / row["filename"])
        )

    status.value = (
        f"<b>Image {current_index + 1}/40</b>"
        f" — {row['image_id']}"
    )

    previous = saved[
        saved["image_id"] == row["image_id"]
    ]

    for name, dropdown in controls.items():
        dropdown.value = "-- select --"

    notes.value = ""

    if not previous.empty:
        record = previous.iloc[0]

        for name, dropdown in controls.items():
            if record[name] in FIELDS[name]:
                dropdown.value = record[name]

        notes.value = str(record["notes"])

def save_next(button):
    global current_index, saved

    if any(
        dropdown.value == "-- select --"
        for dropdown in controls.values()
    ):
        status.value = (
            "<b>Please select all four indicators.</b>"
        )
        return

    row = annotation_images.iloc[current_index]

    record = {
        "image_id": row["image_id"],
        "location_id": row["location_id"],
        **{
            name: dropdown.value
            for name, dropdown in controls.items()
        },
        "notes": notes.value
    }

    saved = saved[
        saved["image_id"] != row["image_id"]
    ]

    saved = pd.concat(
        [saved, pd.DataFrame([record])],
        ignore_index=True
    )

    saved.to_csv(ANNOTATION_FILE, index=False)

    if current_index < len(annotation_images) - 1:
        current_index += 1
        show_image()
    else:
        status.value = (
            f"<b>Annotation complete! "
            f"{len(saved)} images saved.</b>"
        )

save_button.on_click(save_next)

# Resume from the first unannotated image
completed = set(saved["image_id"])
remaining = [
    i for i, image_id in enumerate(annotation_images["image_id"])
    if image_id not in completed
]

if remaining:
    current_index = remaining[0]
    show_image()
    display(
        status,
        image_output,
        *controls.values(),
        notes,
        save_button
    )
else:
    print("All 40 images have already been annotated.")

In [ ]:
import pandas as pd

annotation_path = ANNOTATIONS / "human_annotations.csv"

annotations = pd.read_csv(
    annotation_path,
    keep_default_na=False
)

print("Annotated images:", len(annotations))
print("Unique locations:", annotations["location_id"].nunique())
print("Duplicate image IDs:", annotations["image_id"].duplicated().sum())

fields = [
    "greenery",
    "pedestrian",
    "cycling",
    "motor_vehicle"
]

for field in fields:
    print(f"\n{field.upper()}")
    print(annotations[field].value_counts(dropna=False))

assert len(annotations) == 40
assert annotations["image_id"].is_unique
assert annotations["location_id"].nunique() == 20
assert annotations[fields].notna().all().all()

print("\nBasic validation passed.")